# Demo de Estratégia de Memória Self-Managed do AgentCore

Este notebook demonstra como configurar e usar estratégias de memória self-managed do Amazon Bedrock AgentCore com boto3. A estratégia de memória self-managed permite criar um pipeline personalizado para extração e consolidação de memória, acionado por eventos de conversação.

## Como funciona

1. Configurar triggers: Defina condições de trigger (contagem de mensagens, timeout por inatividade, contagem de tokens) que invocam seu pipeline com base em eventos de memória de curto prazo
2. Receber notificações: O AgentCore publica notificações no seu tópico SNS quando as condições de trigger são atendidas
3. Processar payload: O AgentCore entrega os dados da conversação no seu bucket S3
4. Extrair e armazenar registros de memória: Seu pipeline personalizado recupera o payload e processa as memórias

Para informações detalhadas sobre estratégias de memória self-managed, consulte a [documentação oficial da AWS](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-self-managed-strategies.html#use-self-managed-strategy).

## Visão Geral da Configuração

Este demo irá:
1. Criar a infraestrutura AWS necessária (S3, SNS, SQS, Lambda, IAM roles)
2. Criar uma memória AgentCore com estratégia self-managed
3. Criar eventos de teste para demonstrar o pipeline de processamento de memória
4. Criar um Agent para demonstrar a recuperação e uso de memórias armazenadas
5. Limpar os recursos ao finalizar

## Configuração e Imports

In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
import boto3
import json
import time
import uuid
import os
from datetime import datetime
from aws_utils import AWSUtils

# Configure AWS region
region_name = 'us-west-2'  # Change to your preferred region
aws_utils = AWSUtils(region_name=region_name)

# Read Lambda function code
with open('lambda_function.py', 'r') as f:
    lambda_code = f.read()


In [ ]:
print(lambda_code)

## Passo 1: Criar Bucket S3 para Entrega de Payload

Crie um bucket S3 onde o AgentCore entregará os payloads de conversação quando as condições de trigger forem atendidas.

In [ ]:
# Create S3 bucket with a unique name
bucket_name = aws_utils.create_s3_bucket('agentcore-memory-payloads')
print(f"S3 bucket created: {bucket_name}")

## Passo 2: Criar Tópico SNS para Notificações de Jobs de Memória

Crie um tópico SNS que receberá notificações quando o AgentCore acionar o pipeline de processamento de memória.

In [ ]:
# Create SNS topic
sns_topic_name = f"agentcore-memory-notifications-{int(time.time())}"
sns_topic_arn = aws_utils.create_sns_topic(sns_topic_name)
print(f"SNS topic created: {sns_topic_arn}")

## Passo 3: Criar Fila SQS com Assinatura SNS

Crie uma fila SQS que assina o tópico SNS. Esta fila receberá notificações de jobs de memória que acionarão nossa função Lambda.

In [ ]:
# Create SQS queue and subscribe to SNS topic
queue_name = f"agentcore-memory-queue-{int(time.time())}"
queue_url, queue_arn = aws_utils.create_sqs_queue_with_sns_subscription(queue_name, sns_topic_arn)
print(f"SQS queue created: {queue_url}")

## Passo 4: Criar IAM Roles

Crie duas IAM roles:
1. Para o AgentCore acessar S3 e SNS
2. Para o Lambda acessar S3, SQS e APIs do AgentCore

In [ ]:
# Create IAM role for AgentCore
agentcore_role_name = f"AgentCoreMemoryExecutionRole-{int(time.time())}"
agentcore_role_arn = aws_utils.create_iam_role_for_agentcore(
    agentcore_role_name, 
    bucket_name, 
    sns_topic_arn
)
print(f"AgentCore IAM role created: {agentcore_role_arn}")

# Create IAM role for Lambda
lambda_role_name = f"LambdaMemoryProcessingRole-{int(time.time())}"
lambda_role_arn = aws_utils.create_iam_role_for_lambda(
    lambda_role_name, 
    bucket_name, 
    queue_arn
)
print(f"Lambda IAM role created: {lambda_role_arn}")

## Passo 5: Criar Função Lambda para Processamento de Memória

Crie uma função Lambda que será acionada por mensagens SQS. Esta função irá:
1. Baixar o payload da conversação do S3
2. Extrair memórias usando um modelo Bedrock
3. Armazenar as memórias extraídas de volta no AgentCore

In [ ]:
# Create Lambda function
function_name = f"agentcore-memory-processor-{int(time.time())}"
function_arn = aws_utils.create_lambda_function(
    function_name,
    lambda_role_arn,
    lambda_code
)
print(f"Lambda function created: {function_arn}")

# Add SQS trigger to Lambda
event_source_uuid = aws_utils.add_sqs_trigger_to_lambda(function_name, queue_arn)
print(f"SQS trigger added to Lambda: {event_source_uuid}")

## Passo 6: Criar Memória AgentCore com Estratégia Self-Managed

Crie uma memória AgentCore com uma configuração de estratégia self-managed que utiliza a infraestrutura que configuramos.

In [ ]:

import importlib
import aws_utils
importlib.reload(aws_utils)

# # Create a new instance of AWSUtils with the updated code
aws_utils = aws_utils.AWSUtils(region_name=region_name)

# Create memory with self-managed strategy
memory_name = f"SelfManageMemory{int(time.time())}"
memory_description = "Demo memory using self-managed strategy"

memory_id = aws_utils.create_memory_with_self_managed_strategy(
    memory_name=memory_name,
    memory_description=memory_description,
    role_arn=agentcore_role_arn,
    sns_topic_arn=sns_topic_arn,
    s3_bucket_name=bucket_name,
    message_trigger_count=3,  # Trigger after 3 messages
    token_trigger_count=500,  # Trigger after ~500 tokens
    idle_timeout=300,         # Trigger after 5 minutes of idle time
    historical_window_size=5  # Include 5 previous messages in context
)

print(f"Memory created: {memory_id}")
# print(f"Strategy ID: {strategy_id}")

In [ ]:
def wait_for_memory_to_get_active(memory_id):
    response = aws_utils.agentcore_client_control.get_memory(
        memoryId = memory_id)

    while response['memory']['status'] != 'ACTIVE':
        time.sleep(30)
        response = aws_utils.agentcore_client_control.get_memory(
        memoryId = memory_id)
        print(f"Memory creation status: {response['memory']['status']}")
    return response['memory']['status']

wait_for_memory_to_get_active(memory_id=memory_id)

## Passo 7: Criar Eventos de Teste para Acionar o Pipeline de Memória

Agora vamos criar alguns eventos de teste para acionar o pipeline de memória self-managed. Criaremos eventos suficientes para exceder a contagem de trigger de mensagens.

In [ ]:
actor_id = "test-user-123"

In [ ]:
# Create test events
session_id = aws_utils.create_test_events(
    memory_id=memory_id,
    actor_id=actor_id,
    num_events=6  # This will exceed our message_trigger_count of 3
)

print(f"Created test events with session ID: {session_id}")

In [ ]:
aws_utils.agentcore_client.list_events(
    memoryId = memory_id, 
    actorId = actor_id,
    sessionId = session_id )

## Passo 8: Aguardar o Processamento de Memória

Agora precisamos aguardar a execução do pipeline de processamento de memória. Isso envolve:
1. O AgentCore detectando a condição de trigger (contagem de mensagens excedida)
2. O AgentCore publicando uma notificação no SNS
3. O SNS entregando a mensagem para o SQS
4. O SQS acionando nossa função Lambda
5. O Lambda processando a conversação e armazenando as memórias

Vamos aguardar um pouco e depois verificar se as memórias foram criadas.

In [ ]:
print("Waiting 30 seconds for memory processing to complete...")
time.sleep(30)

## Passo 9: Verificar Registros de Memória

Vamos verificar se nosso pipeline de memória criou registros de memória pesquisando na memória.

In [ ]:
session_id

In [ ]:
# List memory records
namespace=f"/interests/actor/{actor_id}/session/{session_id}/"
def list_memory_records(memory_id, namespace):
    try:
        response = aws_utils.agentcore_client.list_memory_records(
            memoryId=memory_id,
            namespace=namespace
        )
        print(f"Found {len(response.get('memoryRecordSummaries'))} memory records")
        
        # Display the search results
        for idx, result in enumerate(response.get("memoryRecordSummaries")):
            print(f"Memory: {idx}")
            print(f"Content: {result['content']['text']}")
    except Exception as e:
        print(f"Error searching memory: {e}")
list_memory_records(memory_id, namespace)

Observe que os registros acima mostram repetição de interesses do usuário, pois não adicionei nenhuma lógica de consolidação. Portanto, há repetição; com a capacidade de fornecer uma estratégia self-managed, posso definir se quero apenas extração e ingestão. Isso dependerá do seu caso de uso de negócio.

In [ ]:
# Search memory records
def retrieve_memory_records(memory_id, query, topK, namespace):
    try:
        response = aws_utils.agentcore_client.retrieve_memory_records(
            memoryId=memory_id,
            searchCriteria = {
            'searchQuery': query,
            'topK': topK
        },
            namespace=namespace
        )
        print(f"Found {len(response.get('memoryRecordSummaries'))} memory records")
        
        # Display the search results
        for idx, result in enumerate(response.get('memoryRecordSummaries')):
            print(f"\nMemory Record {idx + 1}:")
            print(f"Content: {result['content']['text']}")
    except Exception as e:
        print(f"Error searching memory: {e}")

retrieve_memory_records(memory_id=memory_id, query="food choices for dinner", topK=5, namespace=namespace)

## Passo 10: Criar Eventos de Teste Adicionais com Conteúdo Diferente

Vamos criar mais alguns eventos de teste com conteúdo diferente para acionar outro ciclo de processamento de memória.

In [ ]:
# Create custom test events
session_id = str(uuid.uuid4())
actor_id = "test-user-456"

# Custom events with more specific information
test_events = [
    {
        "user": "I'm trying to eat healthier and have been exploring Mediterranean cuisine lately.",
        "assistant": "That's wonderful! Mediterranean food is both delicious and nutritious. What Mediterranean dishes have you tried so far?"
    },
    {
        "user": "I love Greek salads with feta cheese and olives, and I've been making homemade hummus.",
        "assistant": "Homemade hummus is fantastic! Do you prefer it with tahini or without? And what's your favorite way to serve it?"
    },
    {
        "user": "I always use tahini and like to serve it with fresh vegetables and pita bread. I'm also vegetarian, so I avoid meat.",
        "assistant": "Being vegetarian opens up so many Mediterranean options! Have you tried making stuffed grape leaves or lentil-based dishes?"
    },
    {
        "user": "Not yet, but I'd love to learn. I'm also allergic to shellfish, so I have to be careful with seafood dishes.",
        "assistant": "Good to know about the shellfish allergy. For vegetarian Mediterranean cooking, you might enjoy making moussaka with eggplant or trying some traditional Greek bean dishes. Would you like some recipe suggestions?"
    }
]

# Create events
for idx, event in enumerate(test_events):
    try:
        event_payload = [
            {
                'conversational': {
                    'content': {
                        'text': event['user']
                    },
                    'role': 'USER'
                }
            },
            {
                'conversational': {
                    'content': {
                        'text': event['assistant']
                    },
                    'role': 'ASSISTANT'
                }
            }
        ]

        aws_utils.agentcore_client.create_event(
            memoryId=memory_id,
            actorId=actor_id,
            sessionId=session_id,
            eventTimestamp=int(time.time()),
            payload=event_payload,
            clientToken=str(uuid.uuid4())
        )

        print(f"Created event {idx+1}/{len(test_events)}")
        time.sleep(1)

    except Exception as e:
        print(f"Error creating test event: {e}")

print("\nWaiting 30 seconds for memory processing to complete...")
time.sleep(30)

## Passo 11: Pesquisar Novas Memórias

Agora vamos pesquisar as novas memórias relacionadas a caminhadas e o cachorro do usuário.

In [ ]:
# Search memory records for outdoor activities
namespace=f"/interests/actor/{actor_id}/session/{session_id}/"
retrieve_memory_records(memory_id=memory_id, query="dog pets golden retriever", topK=5, namespace=namespace)

## Passo 12: Criando o agente

Nesta seção, veremos como construir um assistente culinário inteligente usando agentes Strands integrados com a Memória Self-Managed do AgentCore via hooks. Focaremos na memória de longo prazo para preferências alimentares do usuário, restrições dietéticas e histórico gastronômico para fornecer recomendações personalizadas de restaurantes baseadas em conversas anteriores e gostos individuais



In [ ]:
import logging
import json
from typing import Dict
from datetime import datetime
from botocore.exceptions import ClientError

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("customer-support")

# Import required modules
from strands import Agent, tool
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent
from ddgs import DDGS
from bedrock_agentcore.memory import MemoryClient

# Initialize MemoryClient
client = MemoryClient(region_name=region_name)

## Passo 13: Criar Memory Hook Provider para o Assistente Culinário com Memória Self-Managed

Hooks são funções especiais que executam em pontos específicos do ciclo de vida de execução de um agente. Nosso hook provider personalizado utiliza a estratégia de memória self-managed para gerenciar automaticamente o contexto culinário através de:

- **Recuperação de preferências alimentares relevantes** dos registros de memória self-managed
- **Injeção de informações contextuais** sobre restrições dietéticas, preferências de cozinha e histórico gastronômico em novas consultas
- **Salvamento de interações gastronômicas** para referência futura usando operações em lote

Isso cria uma experiência de memória integrada que:
- Recupera automaticamente suas preferências alimentares armazenadas antes de processar cada consulta
- Fornece recomendações de restaurantes com consciência de contexto baseadas no seu histórico gastronômico

A abordagem self-managed nos dá controle total sobre como as preferências alimentares são armazenadas, recuperadas e usadas para aprimorar a experiência de recomendação gastronômica.


In [ ]:
# Helper function to get namespaces from memory strategies list
def get_namespaces(mem_client: MemoryClient, memory_id: str) -> Dict:
    """Get namespace mapping for memory strategies."""
    strategies = mem_client.get_memory_strategies(memory_id)
    return {i["type"]: i["namespaces"][0] for i in strategies}

In [ ]:
class CulinaryAssistantMemoryHooks(HookProvider):
    """Memory hooks for culinary assistant agent"""
    
    def __init__(self, memory_id: str, namespace: str):
        self.memory_id = memory_id
        self.namespace = namespace
    
    def retrieve_food_preferences(self, event: MessageAddedEvent):
        """Retrieve user food preferences before processing dining query"""
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                # Retrieve food preferences using direct API
                response = aws_utils.agentcore_client.retrieve_memory_records(
                    memoryId=self.memory_id,
                    searchCriteria={
                        'searchQuery': user_query,
                        'topK': 5
                    },
                    namespace=self.namespace
                )
                
                memory_records = response.get('memoryRecordSummaries', [])
                
                if memory_records:
                    # Format retrieved preferences
                    preferences_context = []
                    for record in memory_records:
                        content = record.get('content', {}).get('text', '').strip()
                        if content:
                            preferences_context.append(content)
                    
                    # Inject food preferences into the query
                    if preferences_context:
                        context_text = "\n".join(preferences_context)
                        original_text = messages[-1]["content"][0]["text"]
                        messages[-1]["content"][0]["text"] = (
                            f"User Food Preferences:\n{context_text}\n\n{original_text}"
                        )
                        logger.info(f"Retrieved {len(preferences_context)} food preference records")
                
            except Exception as e:
                logger.error(f"Failed to retrieve food preferences: {e}")
    
    def save_dining_interaction(self, event: AfterInvocationEvent):
        """Save dining recommendation interaction after agent response"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # Get last user query and agent response
                user_query = None
                agent_response = None
                
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_query and "toolResult" not in msg["content"][0]:
                        user_query = msg["content"][0]["text"]
                        break
                
                if user_query and agent_response:
                    # Save the interaction using direct API
                    interaction_content = f"Query: {user_query}\nRecommendation: {agent_response}"
                    
                    # You would use create_memory_record API here
                    # aws_utils.agentcore_client.create_memory_record(...)
                    
                    logger.info("Saved dining interaction to memory")
                    
        except Exception as e:
            logger.error(f"Failed to save dining interaction: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        """Register culinary assistant memory hooks"""
        registry.add_callback(MessageAddedEvent, self.retrieve_food_preferences)
        registry.add_callback(AfterInvocationEvent, self.save_dining_interaction)
        logger.info("Culinary assistant memory hooks registered")

## Passo 14: Criar Agente Assistente Culinário

In [ ]:
# Create memory hooks for culinary assistant
print(memory_id)
culinary_hooks = CulinaryAssistantMemoryHooks(memory_id, namespace)

# Create culinary assistant agent
culinary_agent = Agent(
    hooks=[culinary_hooks],
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[],  # Update these tools as needed
    state={"actor_id": actor_id, "session_id": session_id},
    system_prompt="""You are the Culinary Assistant, a sophisticated restaurant recommendation assistant.

PURPOSE:
- Help users discover restaurants based on their preferences
- Remember user preferences throughout the conversation
- Provide personalized dining recommendations

You have access to a Memory tool that enables you to:
- Store user preferences (dietary restrictions, favorite cuisines, budget preferences, etc.)
- Retrieve previously stored information to personalize recommendations"""
)

print("✅ Culinary assistant agent created with memory capabilities")

#### O agente está pronto.

### Vamos testar os Cenários do Assistente Culinário

In [ ]:
response1 = culinary_agent("what are the food choices for Dinner?")
print(f"Support Agent: {response1}")

## Passo 15: Limpeza de Recursos

Agora vamos limpar todos os recursos que criamos para evitar custos desnecessários.

In [ ]:
# Clean up all resources
import importlib
import aws_utils
importlib.reload(aws_utils)

# # Create a new instance of AWSUtils with the updated code
aws_utils = aws_utils.AWSUtils(region_name=region_name)

# # Clean up resources with auto-discovery
aws_utils.cleanup_resources(discover_resources=True)
print("All resources have been cleaned up!")

## Resumo

Neste notebook, demonstramos como:

1. Configurar a infraestrutura AWS necessária para memória self-managed
2. Criar uma memória AgentCore com estratégia self-managed
3. Configurar condições de trigger para processamento de memória
4. Implementar um pipeline de processamento de memória baseado em Lambda
5. Testar o sistema de memória com conversas de exemplo
6. Pesquisar memórias extraídas
7. Criar um agente culinário para testar a memória self-managed
8. Limpar todos os recursos

A estratégia de memória self-managed oferece controle total sobre a extração de memória, permitindo que você construa pipelines personalizados que se adequam ao seu caso de uso específico.